**🪖  Unified Military Analytics**

# Milestone 1: Data Collection and Preparation

This notebook implements Milestone 1 of the project.

The objective is to collect military capability data for multiple countries, structure it into a dataset, and prepare it for further analysis.

## Module 1: Scraping Setup and Execution

In this module, we build a web scraping pipeline to collect military capability metrics.

### Step 1: Create Data Folder

This step creates a local folder named data to store all scraped country CSV files.

In [1]:
import os
os.makedirs("data", exist_ok=True)

### Step 2: Mount Google Drive

This step mounts Google Drive to enable saving files for persistent storage when using Google Colab.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3: Import Required Libraries

We import the necessary libraries required for web scraping and data processing.

In [3]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

### Step 4: Load Country URL

The country URL is defined. Each URL corresponds to a page containing military capability metrics.

In [4]:
url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=india"

### Step 5: Extract Military Data

This step sends a request to the Global Firepower website and extracts India's military data, including power index and detailed metrics.

In [39]:
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

data = {}

title = soup.find("h1")
data["country"] = title.get_text(strip=True)

overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)
    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None

metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

### Step 6: Convert Data into DataFrame

The extracted data is converted into a structured DataFrame and saved as a CSV file.

In [6]:
print("India Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")

df = pd.DataFrame([data])
df.to_csv("data/india_test.csv", index=False)

print("Saved to data/india_test.csv")

India Data Extracted:

country: 2026India Military Strength
power_index_rank: 4
power_index_score: 0.1346
purchasing_power_parity: 3
foreign_exchange/gold: 5
defense_budget: 5
external_debt: 110
square_land_area: 7
coastline_coverage: 91
shared_borders: 129
waterways_(usable): 10
total_population: 2
available_manpower: 2
fit-for-service: 2
reaching_mil_age_annually: 1
tot_mil._personnel_(est.): 4,958,000
active_personnel: 2
reserve_personnel: 8
paramilitary: 2
air_force_personnel*: 3
army_personnel*: 3
navy_personnel*: 4
yearly_mobilization_potential: 23,652,761
aircraft_total: 4
fighters: 4
attack_types: 4
transports_(fixed-wing): 4
trainers: 8
special-mission: 5
tanker_fleet: 11
helicopters: 4
attack_helicopters: 9
tanks: 5
vehicles: 2
self-propelled_artillery: 35
towed_artillery: 3
mlrs_(rocket_artillery): 16
total_assets: 4
total_tonnage: 5
aircraft_carriers: 3
helicopter_carriers: 145
destroyers: 5
frigates: 3
corvettes: 5
submarines: 7
patrol_vessels: 7
mine_warfare: 145
oil_prod

### Step 7: Automate Scraping for Multiple Countries

This step reads multiple URLs from a file and applies the same scraping logic to collect data for all countries.

In [40]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# ✅ Correct page containing ALL countries
url_list_page = "https://www.globalfirepower.com/countries-listing.php"

response = requests.get(url_list_page, timeout=15)
soup = BeautifulSoup(response.text, "lxml")

base_url = "https://www.globalfirepower.com/"

country_links = []

# ✅ Extract ALL country links
for link in soup.find_all("a", href=True):
    href = link["href"]
    if "country-military-strength-detail.php?country_id=" in href:
        full_link = base_url + href
        if full_link not in country_links:
            country_links.append(full_link)

print("Total countries found:", len(country_links))  # should be ~140+

# ✅ Scraping
for i, url in enumerate(country_links):
    try:
        print(f"Processing {i+1}/{len(country_links)}")

        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")
        data = {}

        # Country name
        title = soup.find("h1")
        if title is None:
            continue

        data["country"] = title.get_text(strip=True)

        # Power index
        overview_text = soup.find("span", class_="textNormal")
        if overview_text:
            text = overview_text.get_text(" ", strip=True)
            rank_match = re.search(r"ranked\s+(\d+)", text)
            score_match = re.search(r"score of\s+([\d.]+)", text)

            data["power_index_rank"] = rank_match.group(1) if rank_match else None
            data["power_index_score"] = score_match.group(1) if score_match else None

        # Metrics
        metric_blocks = soup.find_all("div", class_="specsGenContainers")

        for block in metric_blocks:
            label = block.find("span", class_="textLarge")
            value = block.find("span", class_="textWhite")

            if label and value:
                key = (
                    label.get_text(strip=True)
                    .replace(":", "")
                    .lower()
                    .replace(" ", "_")
                )
                data[key] = value.get_text(strip=True)

        # Save file
        df = pd.DataFrame([data])
        df.to_csv(f"data/country_{i}.csv", index=False)

        time.sleep(2)  # prevent blocking

    except Exception as e:
        print(f"Error: {url}")

print("✅ All countries scraped successfully!")

Total countries found: 145
Processing 1/145
Processing 2/145
Processing 3/145
Processing 4/145
Processing 5/145
Processing 6/145
Processing 7/145
Processing 8/145
Processing 9/145
Processing 10/145
Processing 11/145
Processing 12/145
Processing 13/145
Processing 14/145
Processing 15/145
Processing 16/145
Processing 17/145
Processing 18/145
Processing 19/145
Processing 20/145
Processing 21/145
Processing 22/145
Processing 23/145
Processing 24/145
Processing 25/145
Processing 26/145
Processing 27/145
Processing 28/145
Processing 29/145
Processing 30/145
Processing 31/145
Processing 32/145
Processing 33/145
Processing 34/145
Processing 35/145
Processing 36/145
Processing 37/145
Processing 38/145
Processing 39/145
Processing 40/145
Processing 41/145
Processing 42/145
Processing 43/145
Processing 44/145
Processing 45/145
Processing 46/145
Processing 47/145
Processing 48/145
Processing 49/145
Processing 50/145
Processing 51/145
Processing 52/145
Processing 53/145
Processing 54/145
Processing

### Step 8: Merge All CSV Files

This step combines all individual country datasets into a single dataset.

In [41]:
import pandas as pd
import os

folder_path = "data"

all_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

df_list = []

for file in all_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

combined_df.head()

,country,power_index_rank,power_index_score,purchasing_power_parity,foreign_exchange/gold,defense_budget,external_debt,square_land_area,coastline_coverage,shared_borders,...,coal_deficit,coal_proven_reserves,internet_coverage,labor_force,merchant_marine_fleet,ports_/_trade_terminals,airports,roadway_coverage,railway_coverage,total_tonnage
0,2026Benin Military Strength,137,3.8963,116,134,137,33,96,8,52,...,"-164,000 mt",145,47,74,113,46,101,116,109,NaN
1,2026Mexico Military Strength,36,0.6401,13,18,39,119,13,95,89,...,"-8,836,000 mt",42,20,11,28,21,4,12,12,NaN
2,2026Kenya Military Strength,84,1.8588,58,78,81,74,49,31,75,...,"-1,453,000 mt",145,45,29,99,43,18,32,50,NaN
3,2026Japan Military Strength,7,0.1876,5,2,9,139,60,103,999,...,"-169,955,000 mt",53,14,10,4,4,23,6,11,4.0
4,2026Bhutan Military Strength,145,5.7991,140,130,143,14,127,999,24,...,"-101,000 mt",145,13,140,145,145,106,124,145,NaN


### Step 9: Export Final Dataset

This step exports the combined dataset for further processing.

In [38]:
combined_df.to_csv("Military_raw_data.csv", index=False)
print("Combined raw dataset saved as Military_raw_data.csv")

Combined raw dataset saved as Military_raw_data.csv
